# CelebA — Combined Statistical Analysis & Shrinkage Study (method comparison)

Adaptation of `CombinedStatisticalAnalysis_method_comparison.ipynb` (Part A)
and `TestingDDNM.ipynb` (Part B, extended to **all** algorithms) to the
celeb-ddpm study.

**Adaptations from the calo version:**
- Data lives in `[-1, 1]` model space; energy-like diagnostics (missing
  energy, reco/true ratios) use the **intensity** $I = (x+1)/2 \in [0,1]$
  so sums are positive and ratios well-defined.
- Results are RGB: `results.npy` is `(n_img, J, 3, box, box)`; channels are
  treated as extra pixels for statistics, averaged for maps.
- Each run is truncated to its **completed** image count (progress.txt),
  and non-finite images are excluded (counted) — no hardcoded `N_EVENTS`.
- At 256² a materialized `full_samples` array would be ~39 GB per
  algorithm, so full-image ratio / Pearson / event-z use exact algebraic
  decompositions (sample ≡ truth outside the box).
- CRPS uses the sorted O(J log J) estimator (pairwise form would need TBs).
- Label bug fixed (legacy "box {alg}x{alg}" titles).

In [ ]:
import glob, json, os, re
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# ---- config (env-overridable; run from repo root or notebooks/) -----------
DATA_ROOT   = os.environ.get('CELEB_ROOT', 'workdir_celeb')
STUDY_GLOB  = f'{DATA_ROOT}/inpaint_study/*_box*_y*_x*_S*'
IMAGES_GLOB = f'{DATA_ROOT}/generated_images/images_*.npy'

COLORS = {'ddnm':'r', 'repaint':'g', 'ddrm':'b', 'mcg2':'y', 'pigdm2':'m',
          'mcg':'y', 'pigdm':'m'}

def to_intensity(x):
    """[-1,1] model space -> [0,1] intensity (positive, ratio-safe)."""
    return (np.asarray(x, dtype=np.float64) + 1.0) / 2.0

## Load runs (auto-discovered, progress-truncated, NaN-guarded)

In [ ]:
img_files = sorted(glob.glob(IMAGES_GLOB))
assert img_files, f'no truth images found: {IMAGES_GLOB}'
true_images = np.asarray(np.load(img_files[0], mmap_mode='r'))
print('truth images:', img_files[0], true_images.shape)

runs = {}
for run_dir in sorted(glob.glob(STUDY_GLOB)):
    if not os.path.exists(f'{run_dir}/results.npy'):
        continue
    meta = json.load(open(f'{run_dir}/metadata.json'))
    alg  = meta['algorithm']

    res = np.load(f'{run_dir}/results.npy')          # (n, J, C, b, b)
    tru = np.load(f'{run_dir}/truth.npy')            # (n, C, b, b)

    done = res.shape[0]
    if os.path.exists(f'{run_dir}/progress.txt'):
        done = int(open(f'{run_dir}/progress.txt').read().split('/')[0])
    res, tru = res[:done], tru[:done]

    finite = np.isfinite(res).all(axis=(1, 2, 3, 4))
    if (~finite).any():
        print(f'WARNING {alg}: {(~finite).sum()}/{done} non-finite images excluded')
    res, tru = res[finite], tru[finite]
    keep_idx = np.where(finite)[0]

    runs[alg] = {
        'inpaint_dead': to_intensity(res),           # (n, J, C, b, b) in [0,1]
        'truth_dead'  : to_intensity(tru),           # (n, C, b, b)
        'raw_dead'    : res, 'raw_truth': tru,       # [-1,1] (for Part B)
        'idx'         : keep_idx,                    # rows into true_images
        'Y0': meta['y0'], 'X0': meta['x0'], 'BOX': meta['box'], 'meta': meta,
    }
    print(f'{alg:8s} n={len(keep_idx)}  J={res.shape[1]}  '
          f'box={meta["box"]}  S={meta["S"]}')

algs = list(runs)
assert algs, 'no completed runs found'
n_common = min(len(r['idx']) for r in runs.values())
print('common image count for cross-algorithm plots:', n_common)

## Precompute per-image quantities

Full-image quantities via exact decomposition (sample = truth outside the
box): with $t$ = full truth image, $t_b$ = truth box, $s_b$ = sample box
(intensities), the full-sample sum is
$\Sigma s = \Sigma t - \Sigma t_b + \Sigma s_b$, and dot products
decompose the same way for the Pearson coefficient.

In [ ]:
for alg, r in runs.items():
    imgs = to_intensity(true_images[r['idx']])       # (n, C, H, W)
    Y0, X0, B = r['Y0'], r['X0'], r['BOX']

    r['true_full_sum']  = imgs.sum(axis=(1, 2, 3))                  # (n,)
    r['true_full_mean'] = imgs.mean(axis=(1, 2, 3))                 # (n,)
    r['true_box_sum']   = r['truth_dead'].sum(axis=(1, 2, 3))       # (n,)
    r['reco_box_sum']   = r['inpaint_dead'].sum(axis=(2, 3, 4))     # (n, J)
    r['reco_full_sum']  = (r['true_full_sum'][:, None]
                           - r['true_box_sum'][:, None]
                           + r['reco_box_sum'])                     # (n, J)

    # pieces for full-image Pearson (uncentered dots + means)
    D = imgs[0].size
    r['D'] = D
    r['dot_tt']   = (imgs ** 2).sum(axis=(1, 2, 3))                          # t.t
    r['dot_tbtb'] = (r['truth_dead'] ** 2).sum(axis=(1, 2, 3))              # tb.tb
    r['dot_tbsb'] = (r['truth_dead'][:, None] * r['inpaint_dead']).sum(axis=(2, 3, 4))
    r['dot_sbsb'] = (r['inpaint_dead'] ** 2).sum(axis=(2, 3, 4))
    del imgs
print('precomputed.')

## Part A / Diagnostic 1 — Missing-intensity ratio (accuracy)

In [ ]:
def binned_weighted_stats(x, y, dy, bins):
    x, y, dy, bins = map(np.asarray, (x, y, dy, bins))
    idx = np.digitize(x, bins) - 1
    nb  = len(bins) - 1
    centers = 0.5 * (bins[:-1] + bins[1:])
    mean = np.full(nb, np.nan); err = np.full(nb, np.nan)
    cnt  = np.zeros(nb, int)
    for i in range(nb):
        m = (idx == i) & (dy > 0)
        cnt[i] = m.sum()
        if cnt[i] == 0: continue
        w = 1.0 / dy[m] ** 2
        mean[i] = np.sum(w * y[m]) / w.sum()
        err[i]  = np.sqrt(1.0 / w.sum())
    return centers, mean, err, cnt

def binned_mean_stats(x, y, bins):
    x, y, bins = map(np.asarray, (x, y, bins))
    idx = np.digitize(x, bins) - 1
    nb  = len(bins) - 1
    centers = 0.5 * (bins[:-1] + bins[1:])
    mean = np.full(nb, np.nan); err = np.full(nb, np.nan)
    cnt  = np.zeros(nb, int)
    for i in range(nb):
        m = idx == i
        cnt[i] = m.sum()
        if cnt[i] == 0: continue
        v = y[m]
        mean[i] = v.mean()
        err[i]  = v.std(ddof=1) / np.sqrt(cnt[i]) if cnt[i] > 1 else 0.0
    return centers, mean, err, cnt

# per-image mean truth intensity in the box spans a narrow range for faces:
lo = min(r['truth_dead'].mean(axis=(1,2,3)).min() for r in runs.values())
hi = max(r['truth_dead'].mean(axis=(1,2,3)).max() for r in runs.values())
bins = np.linspace(0.95*lo, 1.05*hi, 13)

plt.figure()
for alg, r in runs.items():
    ratio = r['true_box_sum'][:, None] / r['reco_box_sum']
    centers, mean, sig, _ = binned_weighted_stats(
        r['truth_dead'].mean(axis=(1,2,3)), ratio.mean(axis=1),
        ratio.std(axis=1), bins)
    plt.errorbar(centers, mean, yerr=sig, fmt=COLORS.get(alg,'k')+'.',
                 capsize=4, label=alg)
plt.axhline(1.0, ls='--', color='k', lw=0.8)
plt.xlabel('mean true box intensity')
plt.ylabel('true / reco box intensity')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

## Part A / Diagnostic 2 — Full-image reco/true intensity ratio

In [ ]:
plt.figure()
for alg, r in runs.items():
    ratio = r['reco_full_sum'] / r['true_full_sum'][:, None]
    centers, mean, sig, _ = binned_weighted_stats(
        r['true_full_mean'], ratio.mean(axis=1), ratio.std(axis=1) + 1e-12,
        np.linspace(0.95*r['true_full_mean'].min(),
                    1.05*r['true_full_mean'].max(), 13))
    plt.errorbar(centers, mean, yerr=sig, fmt=COLORS.get(alg,'k')+'.',
                 capsize=4, label=alg)
plt.axhline(1.0, ls='--', color='k', lw=0.8)
plt.xlabel('mean true intensity (full image)')
plt.ylabel('reco / true full-image intensity')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

## Part A / Diagnostic 3 — Full-image Pearson (algebraic, exact)

Same quantity as the calo notebook, computed from the box decomposition
without materializing full samples.

In [ ]:
plt.figure()
for alg, r in runs.items():
    n, J = r['reco_box_sum'].shape
    D    = r['D']
    t_mean = r['true_full_mean']                                   # (n,)
    s_sum  = r['reco_full_sum']                                    # (n,J)
    s_mean = s_sum / D
    # uncentered dots via decomposition
    dot_ts = (r['dot_tt'][:, None] - r['dot_tbtb'][:, None] + r['dot_tbsb'])
    dot_ss = (r['dot_tt'][:, None] - r['dot_tbtb'][:, None] + r['dot_sbsb'])
    t_sum  = r['true_full_sum']
    cov  = dot_ts - D * t_mean[:, None] * s_mean
    vt   = r['dot_tt'] - D * t_mean ** 2
    vs   = dot_ss - D * s_mean ** 2
    pearson = cov / np.sqrt(np.clip(vt[:, None] * vs, 1e-30, None))
    centers, mean, sig, _ = binned_mean_stats(
        t_mean, pearson.mean(axis=1),
        np.linspace(0.95*t_mean.min(), 1.05*t_mean.max(), 13))
    plt.errorbar(centers, mean, yerr=sig, fmt=COLORS.get(alg,'k')+'.',
                 capsize=4, label=alg)
plt.xlabel('mean true intensity (full image)')
plt.ylabel('mean full-image Pearson coefficient')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

## Part A / Diagnostic 4 — z-score diagnostics (dead region)

Computed directly on the dead box (outside it samples equal truth, so those
pixels contributed nothing in the calo version either).

In [ ]:
def zscore_diag(r):
    t, s = r['truth_dead'], r['inpaint_dead']
    mu  = s.mean(axis=1)
    sd  = s.std(axis=1, ddof=1)
    sd[sd < 1e-12] = np.nan
    z   = (t - mu) / sd
    # event-level z on the box mean
    bm_s  = s.mean(axis=(2, 3, 4))
    ev_mu, ev_sd = bm_s.mean(axis=1), bm_s.std(axis=1, ddof=1)
    ev_sd[ev_sd < 1e-12] = np.nan
    ev_z = (t.mean(axis=(1, 2, 3)) - ev_mu) / ev_sd
    return {'event_z': ev_z,
            'rms_z': np.sqrt(np.nanmean(z**2, axis=(1,2,3))),
            'frac_lt1': np.nanmean(np.abs(z) < 1, axis=(1,2,3))}

zres = {alg: zscore_diag(r) for alg, r in runs.items()}

for key, ylabel, ref in [('event_z', 'event z (box mean)', 0.0),
                         ('rms_z', 'pixel RMS(z)', 1.0),
                         ('frac_lt1', 'fraction |z| < 1', 0.6827)]:
    plt.figure(figsize=(7, 4.5))
    for alg, res in zres.items():
        x = runs[alg]['true_full_mean']
        centers, mean, sig, _ = binned_mean_stats(
            x, res[key], np.linspace(0.95*x.min(), 1.05*x.max(), 11))
        plt.errorbar(centers, mean, yerr=sig, fmt=COLORS.get(alg,'k')+'.',
                     capsize=4, label=alg)
    plt.axhline(ref, ls='--', color='k')
    plt.xlabel('mean true intensity (full image)')
    plt.ylabel(ylabel)
    plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

## Part A / Diagnostic 5 — Spatial bias maps (channel-averaged)

In [ ]:
n_a = len(algs)
fig, axes = plt.subplots(1, n_a, figsize=(4*n_a, 3.6), squeeze=False)
for a, alg in zip(axes[0], algs):
    r = runs[alg]
    bias = (r['inpaint_dead'].mean(axis=1) - r['truth_dead']) \
        .mean(axis=(0, 1))                       # avg events + channels -> (b,b)
    vmax = np.abs(bias).max()
    im = a.imshow(bias, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    a.set_title(f'{alg}', fontsize=10)
    plt.colorbar(im, ax=a, fraction=0.046)
fig.suptitle('spatial bias: mean(inpainted − truth), intensity units')
plt.tight_layout(); plt.show()

## Part A / Diagnostic 6 — PIT histograms

In [ ]:
fig, axes = plt.subplots(1, len(algs), figsize=(3.2*len(algs), 3),
                         squeeze=False)
for a, alg in zip(axes[0], algs):
    r = runs[alg]
    pit = np.mean(r['inpaint_dead'] <= r['truth_dead'][:, None], axis=1).ravel()
    a.hist(pit, bins=20, range=(0, 1), density=True, alpha=0.7,
           color=COLORS.get(alg, 'k'))
    a.axhline(1.0, color='k', ls='--')
    a.set_title(alg, fontsize=10); a.set_xlabel('PIT')
axes[0][0].set_ylabel('density')
plt.tight_layout(); plt.show()

## Part A / Diagnostic 7 — Empirical coverage

In [ ]:
def empirical_coverage(t, s, levels=(0.5, 0.8, 0.9, 0.95)):
    out = {}
    for level in levels:
        lo = np.quantile(s, (1-level)/2, axis=1)
        hi = np.quantile(s, 1-(1-level)/2, axis=1)
        out[level] = float(((t >= lo) & (t <= hi)).mean())
    return out

print('Empirical coverage (nominal -> observed):')
for alg, r in runs.items():
    cov = empirical_coverage(r['truth_dead'], r['inpaint_dead'])
    print(f'  {alg:8s}: ' + ', '.join(f'{l:.0%}->{o:.1%}' for l, o in cov.items()))

## Part A / Diagnostic 8 — CRPS and sharpness

CRPS via the sorted-sample estimator,
$\mathrm{CRPS} = \overline{|s - t|} - \frac{1}{2}\,\mathbb{E}|s - s'|$
with $\mathbb{E}|s-s'| = \frac{2}{J^2}\sum_i (2i - J - 1)\, s_{(i)}$
(O(J log J), no pairwise memory blowup).

In [ ]:
def crps_sorted(t, s):
    """t: (..., ) truth; s: (..., J) samples on last axis."""
    J = s.shape[-1]
    term1 = np.abs(s - t[..., None]).mean(axis=-1)
    ss = np.sort(s, axis=-1)
    i  = np.arange(1, J + 1)
    term2 = (2 / (J * J)) * ((2 * i - J - 1) * ss).sum(axis=-1)
    return term1 - 0.5 * term2

crps_by, sharp_by = {}, {}
for alg, r in runs.items():
    s = np.moveaxis(r['inpaint_dead'], 1, -1)      # (n, C, b, b, J)
    crps_by[alg]  = float(crps_sorted(r['truth_dead'], s).mean())
    sharp_by[alg] = float(r['inpaint_dead'].std(axis=1).mean())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.bar(list(crps_by), list(crps_by.values()),
       color=[COLORS.get(a, 'k') for a in crps_by])
a1.set_ylabel('mean CRPS [intensity]'); a1.set_title('CRPS (lower = better)')
a2.bar(list(sharp_by), list(sharp_by.values()),
       color=[COLORS.get(a, 'k') for a in sharp_by])
a2.set_ylabel('mean ensemble std'); a2.set_title('sharpness (context for CRPS)')
plt.tight_layout(); plt.show()
for alg in crps_by:
    print(f'  {alg:8s}: CRPS={crps_by[alg]:.5f}  sharpness={sharp_by[alg]:.5f}')

## Part A / Diagnostic 9 — Example reconstructions (RGB)

In [ ]:
EVENT_IDX = 0
for alg, r in runs.items():
    i_img = r['idx'][EVENT_IDX]
    Y0, X0, B = r['Y0'], r['X0'], r['BOX']
    truth = to_intensity(true_images[i_img]).transpose(1, 2, 0)   # (H,W,3)
    mean_box = r['inpaint_dead'][EVENT_IDX].mean(axis=0).transpose(1, 2, 0)
    one_box  = r['inpaint_dead'][EVENT_IDX, 0].transpose(1, 2, 0)

    mean_img = truth.copy(); mean_img[Y0:Y0+B, X0:X0+B] = mean_box
    one_img  = truth.copy(); one_img[Y0:Y0+B, X0:X0+B]  = one_box
    resid    = (mean_img - truth).mean(axis=2)

    fig, ax = plt.subplots(1, 4, figsize=(14, 3.6))
    for a, (img, ttl) in zip(ax, [(truth, 'truth'), (mean_img, 'ensemble mean'),
                                  (one_img, 'one sample'), (resid, 'mean − truth')]):
        if ttl == 'mean − truth':
            v = np.abs(img).max() + 1e-9
            im = a.imshow(img, cmap='RdBu_r', vmin=-v, vmax=v)
            plt.colorbar(im, ax=a, fraction=0.046)
        else:
            a.imshow(np.clip(img, 0, 1))
        a.add_patch(plt.Rectangle((X0-.5, Y0-.5), B, B, fill=False,
                                  edgecolor='lime', lw=1.5))
        a.set_title(ttl, fontsize=10); a.axis('off')
    fig.suptitle(f'{alg} — image {i_img}')
    plt.tight_layout(); plt.show()

# Part B — Shrinkage study, all algorithms

Adaptation of `TestingDDNM.ipynb`: per-image **box-mean** of truth vs
posterior mean (raw $[-1,1]$ model space, matching how the calo version
used GeV directly), one panel per algorithm with the regression fit, plus a
combined summary.  Slope < 1 is the expected posterior-mean shrinkage; the
comparison across algorithms (and against the calo numbers) is the point.

In [ ]:
fig, axes = plt.subplots(1, len(algs), figsize=(4.2*len(algs), 4),
                         squeeze=False)
summary = {}
for a, alg in zip(axes[0], algs):
    r = runs[alg]
    t_m = r['raw_truth'].mean(axis=(1, 2, 3))                 # (n,)
    f_m = r['raw_dead'].mean(axis=(1, 2, 3, 4))               # posterior mean
    f_s = r['raw_dead'].mean(axis=(2, 3, 4)).std(axis=1, ddof=1)

    res = stats.linregress(t_m, f_m)
    summary[alg] = res

    a.errorbar(t_m, f_m, yerr=f_s, fmt='o', ms=3, alpha=0.5, capsize=2,
               color=COLORS.get(alg, 'k'))
    xx = np.linspace(t_m.min(), t_m.max(), 50)
    a.plot(xx, res.intercept + res.slope*xx, 'k-', lw=2,
           label=f'y={res.slope:.2f}x+{res.intercept:.2f}\nR²={res.rvalue**2:.2f}')
    lim = [min(t_m.min(), f_m.min()), max(t_m.max(), f_m.max())]
    a.plot(lim, lim, 'r--', lw=1, label='y = x')
    a.set_title(alg); a.set_xlabel('truth box mean')
    a.grid(alpha=0.3); a.legend(fontsize=8)
axes[0][0].set_ylabel('posterior-mean box mean')
plt.tight_layout(); plt.show()

print('Shrinkage summary (slope of posterior mean vs truth, box means):')
for alg, res in summary.items():
    print(f'  {alg:8s}: slope={res.slope:.3f}  intercept={res.intercept:.3f}  '
          f'R²={res.rvalue**2:.3f}')

In [ ]:
# 2D histograms (the TestingDDNM view), one per algorithm
fig, axes = plt.subplots(1, len(algs), figsize=(4*len(algs), 3.6),
                         squeeze=False)
for a, alg in zip(axes[0], algs):
    r = runs[alg]
    t_m = r['raw_truth'].mean(axis=(1, 2, 3))
    f_m = r['raw_dead'].mean(axis=(1, 2, 3, 4))
    h = a.hist2d(t_m, f_m, bins=15, cmap='viridis')
    lim = [min(t_m.min(), f_m.min()), max(t_m.max(), f_m.max())]
    a.plot(lim, lim, 'r--', lw=1)
    a.set_title(alg, fontsize=10); a.set_xlabel('truth box mean')
    plt.colorbar(h[3], ax=a, fraction=0.046)
axes[0][0].set_ylabel('posterior-mean box mean')
plt.tight_layout(); plt.show()